In [1]:
import pickle as pkl

import numpy as np
import pandas as pd
from tqdm import tqdm

from normalize_vendor import normalize_vendor

#### Objective:
Experiment with techniques for normalizing vendors. The function `normalize_vendor` above started its life in this notebook.

#### Depends on:

In [2]:
fname = "./data/01-parsed-amounts-and-dates.parquet"

#### Generates:

In [ ]:
vendor_to_idx_map_fname = "./data/norm_vendors_list_to_idx_map.pkl"

A simple map to make going from normalized vendor names to indexes consistent.

------------------

In [3]:
df = pd.read_parquet(fname)

In [4]:
df.head()

,cohort,agency_number,agency_name,holder_last_name,holder_first_initial,amount,vendor,mcc,parsed_amount,transaction_date,posted_date
0,201307,1000,OKLAHOMA STATE UNIVERSITY,Mason,C,890,NACAS,CHARITABLE AND SOCIAL SERVICE ORGANIZATIONS,890.00,2013-07-30,2013-07-31
1,201307,1000,OKLAHOMA STATE UNIVERSITY,Mason,C,368.96,SHERATON HOTEL,SHERATON,368.96,2013-07-30,2013-07-31
2,201307,1000,OKLAHOMA STATE UNIVERSITY,Massey,J,165.82,SEARS.COM 9300,DIRCT MARKETING/DIRCT MARKETERS--NOT ELSEWHERE...,165.82,2013-07-29,2013-07-31
3,201307,1000,OKLAHOMA STATE UNIVERSITY,Massey,T,96.39,WAL-MART #0137,"GROCERY STORES,AND SUPERMARKETS",96.39,2013-07-30,2013-07-31
4,201307,1000,OKLAHOMA STATE UNIVERSITY,Mauro-Herrera,M,125.96,STAPLES DIRECT,"STATIONERY, OFFICE SUPPLIES, PRINTING AND WRIT...",125.96,2013-07-30,2013-07-31


In [5]:
vendors_str = list(set(df.vendor))
len(vendors_str)

86727

In [6]:
for round in range(5):
    print("|", "| |".join(np.random.choice(vendors_str, 5)), "|", sep="")

|SOONER STEEL SALES| |TWTIRE OKC AUTOMOTIVE| |DELTA AIR   0067458662538| |USPTO| |APPLIED STEMCELL INC|
|RC PLANET.COM| |AMERICAN AI 0010268381062| |APPA BB&T| |MARIANNES RENTALS| |JOURNYHSE   PIOTROWSKI|
|PLS DIGITAL TUTORS| |OFFICEMAX CT IN#105070| |CITY OF MANGUM| |AGENT FEE   8900615166346| |JOURNYHSE   BARTLEYL|
|EB  OKIPC 2014 FIELD M| |AMERICAN AI 0017369561105| |SOUTHWES    5262162812891| |MXTOOLS| |EMIRATES AI 1767352917931|
|MICRO STAR TECHNOLOGIES| |STAPLES       00107391| |UNITED      0167453716617| |HILTON HOTELS TAPATIO| |COURTYARD BY MARRIOTT NRM|


In [7]:
with open("./data/raw_vendors_list.pkl", "wb") as f:
    pkl.dump(vendors_str, f)

In [8]:
samples = 10

# search_filter = list(filter(lambda x: "-" in x, vendors_str))

for s in np.random.choice(vendors_str, samples):
    print(s, "->", normalize_vendor(s))

DELTA AIR   0067310420801 -> DELTA AIR
FEDEX 770128848270 -> FEDEX
APPFIGURES -> APPFIGURES
WAL-MART #4615 -> WALMART
DELTA AIR   0067290276548 -> DELTA AIR
AMERICAN AI 0017445409374 -> AMERICAN AI
JOURNYHSE   REAVIS -> JOURNYHSE
SOUTHWES    5262186754652 -> SOUTHWES
HILTON GARDEN INN F&B -> HILTON GARDEN INN F&B
AGENT FEE   8900594806478 -> AGENT FEE


In [9]:
norm_vendors_str = sorted(
    list(set(filter(lambda x: len(x) > 0, map(normalize_vendor, vendors_str))))
)

In [10]:
len(norm_vendors_str)

30647

In [11]:
len(norm_vendors_str) / len(vendors_str)

0.35337322863698734

In [12]:
with open("./data/norm_vendors_list.pkl", "wb") as f:
    pkl.dump(norm_vendors_str, f)

In [13]:
norm_vendors_str[0]

'A & N'

In [14]:
# find common words

hit_count = {}

for name in norm_vendors_str:
    for w in name.split(" "):
        if w not in hit_count:
            hit_count[w] = 0
        hit_count[w] += 1

In [15]:
print(
    {
        k: v
        for k, v in sorted(hit_count.items(), key=lambda item: item[1], reverse=True)
        if v > 50
    }
)

{'THE': 701, 'AND': 559, 'SUPPLY': 557, 'INN': 475, 'AMERICAN': 347, 'HOTEL': 346, 'PRODUCTS': 281, 'OKLAHOMA': 260, 'SYSTEMS': 241, 'CITY': 220, 'MEDICAL': 214, 'SUITES': 210, 'SERVICE': 207, 'CENTER': 205, 'FACEBK': 186, 'SERVICES': 185, 'FOR': 181, 'MARRIOTT': 180, 'USA': 176, 'INT': 174, 'AUTO': 170, '': 169, 'EQUIPMENT': 169, 'STORE': 165, 'GROUP': 159, 'NATIONAL': 157, 'ADJ': 156, 'CLAIM': 152, 'SALES': 151, 'HOTELS': 148, 'WESTERN': 146, 'ASSOC': 144, 'TULSA': 144, 'OFFICE': 140, 'INDUSTRIES': 138, 'ELECTRIC': 137, 'HOLIDAY': 134, 'AIR': 130, 'EXPRESS': 130, 'HILTON': 129, 'ONLINE': 126, 'LTD': 126, 'SOLUTIONS': 126, 'SHOP': 123, 'PARTS': 120, 'SOCIETY': 117, 'BEST': 117, 'TIRE': 115, 'TECHNOLOGIES': 114, 'SOFTWARE': 107, 'TECH': 100, 'BIO': 100, 'OKC': 100, 'HYATT': 96, 'ACT': 95, 'CAFE': 95, 'DIRECT': 94, 'RESTAURANT': 93, 'NEW': 93, 'SAFETY': 91, 'HEALTH': 90, 'GRILL': 90, 'SPORTS': 89, 'TECHNOLOGY': 89, 'MEDIA': 88, 'PRO': 88, 'INNS': 88, 'HOUSE': 85, 'WEB': 84, 'AMERICA': 8

In [16]:
print([s for s in norm_vendors_str if "COM" in s][:20])

['A&M COMMERCE ACADEMICS', 'AACH DOCCOM', 'AAIRETEMP COMFORT SPEC', 'AAMCOMP', 'ABSOLUTE COMPUTERS', 'ACADEMIC COMMUNICATION', 'ACE COMPUTERS', 'ACOPIAN TECHNICAL COMPAN', 'ACT AMERI CAN TELECOM', 'ACT OKLAHOMA TAX COMMI', 'ACTIVITY CONNECTIONCOM', 'ADA AREA CHAMBER COMME', 'ADHESIVE COMPOUNDERS', 'ADVANSTAR COMMUNICATIONS', 'AERO PRODUCTS COMPONENT', 'AEROSPACE COMPOSITE PROD', 'AFL TELECOMMUNICATIONS', 'AGES COMPUTERS', 'AGRI COMMUNICATIONS', 'AIR COMPRESSOR PRODUCTS']


In [17]:
df[df.vendor.str.contains("FACEBK")]

,cohort,agency_number,agency_name,holder_last_name,holder_first_initial,amount,vendor,mcc,parsed_amount,transaction_date,posted_date
97,201307,1000,OKLAHOMA STATE UNIVERSITY,Gross,M,15.14,FACEBK MZPXH4AKR2,ADVERTISING SERVICES,15.14,2013-07-27,2013-07-29
981,201307,1000,OKLAHOMA STATE UNIVERSITY,Whitmore,D,26.88,FACEBK FSGWH4NQD2,ADVERTISING SERVICES,26.88,2013-07-27,2013-07-29
12362,201308,1000,OKLAHOMA STATE UNIVERSITY,Mason,K,65.16,FACEBK F9KQS4ARB2,ADVERTISING SERVICES,65.16,2013-08-15,2013-08-16
14248,201308,1000,OKLAHOMA STATE UNIVERSITY,Mason,K,34.84,FACEBK U6HAS4ARB2,ADVERTISING SERVICES,34.84,2013-08-14,2013-08-15
14631,201308,1000,OKLAHOMA STATE UNIVERSITY,Whitmore,D,30.68,FACEBK 9PVRQ4WQD2,ADVERTISING SERVICES,30.68,2013-08-01,2013-08-01
...,...,...,...,...,...,...,...,...,...,...,...
432086,201405,77000,UNIV. OF OKLA. HEALTH SCIENCES CENTER,WILSON,S,100.63,FACEBK 9AMHU52D42,ADVERTISING SERVICES,100.63,2014-05-19,2014-05-20
432248,201405,77000,UNIV. OF OKLA. HEALTH SCIENCES CENTER,WILSON,S,100.2,FACEBK V65UV5SC42,ADVERTISING SERVICES,100.20,2014-05-23,2014-05-23
435181,201406,77000,UNIV. OF OKLA. HEALTH SCIENCES CENTER,WILSON,S,4.35,FACEBK 6C8C262C42,ADVERTISING SERVICES,4.35,2014-05-31,2014-06-02
435339,201406,77000,UNIV. OF OKLA. HEALTH SCIENCES CENTER,WILSON,S,101.03,FACEBK FKPCW52D42,ADVERTISING SERVICES,101.03,2014-05-31,2014-06-02


In [18]:
[(s, len(s)) for s in norm_vendors_str if len(s) < 3]

[]

In [19]:
normalize_vendor("WWW.NCFR.ORG")

'NCFR ORG'

In [20]:
vendors_idx_to_name = {i: name for i, name in enumerate(norm_vendors_str)}
vendors_name_to_idx = {name: i for i, name in enumerate(norm_vendors_str)}

In [21]:
with open(vendor_to_idx_map_fname, "wb") as f:
    pkl.dump(
        {
            "idx_to_norm_name": vendors_idx_to_name,
            "norm_name_to_idx": vendors_name_to_idx,
        },
        f,
    )